<a href="https://colab.research.google.com/github/sarthakkad15/UCS420/blob/main/Assignment_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCS420 Lab Assignment 7 — NLP Text Preprocessing

In [5]:
ROLL_NO = 1024170117        # <-- your full roll number
NAME = "Sarthak" # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [6]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [7]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [8]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024170117   Name: Sarthak   Fingerprint: 21B9B294   Tickets: 14

Ticket_ID                                                                Ticket_Text
      T01                       The printer has stopped syncing. Can you check this?
      T02                                       Python 3.12 was installed yesterday.
      T03 I am unable to use the password during the lab. Is there another solution?
      T04                                The VPN stopped responding. This is urgent.
      T05    I can't open the laptop before the assignment deadline. This is urgent.
      T06       I cannot access the MATLAB while uploading the file. This is urgent.
      T07                                  The WiFi stopped responding. Please help.
      T08                                                My roll no. is 1023-03-021.
      T09                                The state-of-the-art lab (Lab-3) is closed.
      T10                                      It's not the laptop; it'

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [9]:
results = []
processed_tickets = {}

for _, row in df.iterrows():
    ticket_id = row["Ticket_ID"]
    text = row["Ticket_Text"]

    processed = process_ticket(text)
    processed_tickets[ticket_id] = processed

    results.append({
        "Ticket_ID": ticket_id,
        "Sentences": len(processed["sentences"]),
        "Tokens": len(processed["tokens"]),
        "Punctuation Tokens": len(processed["punct"]),
        "Stopwords": len(processed["stops"]),
        "Unique Tokens": len(set(processed["tokens"])),
        "Final Tokens": len(processed["final"])
    })

summary_df = pd.DataFrame(results)

total_row = {
    "Ticket_ID": "TOTAL",
    "Sentences": summary_df["Sentences"].sum(),
    "Tokens": summary_df["Tokens"].sum(),
    "Punctuation Tokens": summary_df["Punctuation Tokens"].sum(),
    "Stopwords": summary_df["Stopwords"].sum(),
    "Unique Tokens": summary_df["Unique Tokens"].sum(),
    "Final Tokens": summary_df["Final Tokens"].sum()
}

summary_df = pd.concat(
    [summary_df, pd.DataFrame([total_row])],
    ignore_index=True
)

display(summary_df)

selected_ticket_id = df.iloc[0]["Ticket_ID"]
selected = processed_tickets[selected_ticket_id]

print("Selected Ticket ID:", selected_ticket_id)
print("\nOriginal text:")
print(df.iloc[0]["Ticket_Text"])

print("\nSentences:")
print(selected["sentences"])

print("\nTokens:")
print(selected["tokens"])

print("\nPunctuation tokens:")
print(selected["punct"])

print("\nStopwords:")
print(selected["stops"])

print("\nUnique tokens:")
print(sorted(set(selected["tokens"])))

print("\nFinal tokens:")
print(selected["final"])


,Ticket_ID,Sentences,Tokens,Punctuation Tokens,Stopwords,Unique Tokens,Final Tokens
0,T01,2,11,2,5,11,4
1,T02,1,6,1,1,6,4
2,T03,2,16,2,8,15,6
3,T04,2,9,2,3,8,4
4,T05,2,15,2,6,13,7
5,T06,2,15,2,8,13,5
6,T07,2,8,2,1,7,5
7,T08,2,7,2,3,6,2
8,T09,1,9,3,2,9,4
9,T10,1,11,2,5,8,4


Selected Ticket ID: T01

Original text:
The printer has stopped syncing. Can you check this?

Sentences:
['The printer has stopped syncing.', 'Can you check this?']

Tokens:
['the', 'printer', 'has', 'stopped', 'syncing', '.', 'can', 'you', 'check', 'this', '?']

Punctuation tokens:
['.', '?']

Stopwords:
['the', 'has', 'can', 'you', 'this']

Unique tokens:
['.', '?', 'can', 'check', 'has', 'printer', 'stopped', 'syncing', 'the', 'this', 'you']

Final tokens:
['printer', 'stopped', 'syncing', 'check']


**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:*  Some final tokens may not be complete English words, such as n't, 's. These arise because contractions are split into separate tokens during word tokenization. Since the preprocessing pipeline removes punctuation tokens and stopwords but does not perform stemming, lemmatization, or contraction normalization, these fragments can remain in the final token list.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

import re

raw_texts = df["Ticket_Text"].tolist()
all_texts = raw_texts + TEST_LINES

patterns = {
    "Words longer than 5 letters": r"\b[A-Za-z]{6,}\b",
    "Numbers": r"\b\d+(?:\.\d+)*\b",
    "Capitalized words": r"\b[A-Z][a-zA-Z]*\b",
    "Words starting with a vowel": r"\b[AEIOUaeiou][A-Za-z]*\b"
}

for name, pattern in patterns.items():
    print(f"\n{'=' * 60}")
    print(name)
    print("=" * 60)

    for i, text in enumerate(all_texts, start=1):
        matches = re.findall(pattern, text)
        print(f"Text {i}: {matches}")

def redact_text(text):
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "<EMAIL>",
        text
    )

    text = re.sub(
        r"\b(?:https?://|www\.)[^\s]+",
        "<URL>",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"(?<!\w)(?:\+?\d{1,3}[- ]?)?(?:\d{5}[- ]?\d{5}|\d{3,5}[- ]\d{4,5})(?!\w)",
        "<PHONE>",
        text
    )

    return text

print("\n\n========== REDACTED TEXTS ==========")

for i, text in enumerate(all_texts, start=1):
    print(f"Text {i}: {redact_text(text)}")

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:*

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
# Your code

**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:*

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
# Your code

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:*

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Your code

**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:*